In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "VariableComparisons")
dataType = "VariableHistograms"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

Region = "TRACER"; Case = "WET"; spinup_hours = "0"
# Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"
# Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
# Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

# Region = "Hawaii"; Case = "WET"; spinup_hours = "12"; spinup_hours="-16"
# Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def GetVarData(data, varName):
    if varName == "qc+qi":
        varName_split = varName.split("+")
        varData = data[varName_split[0]] + data[varName_split[1]]
    else:
        varData = data[varName]

    return varData

In [ ]:
def MakeHistogram(varData, varBins, altitude_levels,altitude_data, altitude_bins):
    """
    Simple fast 2-D histogram generator for gridded data (CFAD-style),
    matching orientation and behavior of the original version.
    """
    # Mask invalid values (NaN or inf)
    field_data = np.ma.masked_invalid(np.asarray(varData))

    # Broadcast altitude levels to 3D grid shape
    if altitude_data is None:
        nz, ny, nx = field_data.shape
        altitude_data = np.repeat(altitude_levels[:, None, None], ny, axis=1)
        altitude_data = np.repeat(altitude_data, nx, axis=2)

    # Apply the same mask to altitude data
    altitude_data = np.ma.masked_where(field_data.mask, altitude_data)

    # Match the original orientation: X = variable, Y = altitude
    hist2d_raw, var_edges, z_edges = np.histogram2d(
        field_data.compressed(),      # variable values first
        altitude_data.compressed(),   # altitude values second
        bins=[varBins, altitude_bins] # same order as original
    )
    return hist2d_raw, var_edges, z_edges
    
def NormalizeHistogram(hist2d_raw):
    level_sums = hist2d_raw.sum(axis=1, keepdims=True)
    level_sums[level_sums == 0] = 1
    hist2d_norm = hist2d_raw / level_sums
    return hist2d_norm * 100

In [ ]:
def RunHistogram(data_NSSL,data_TEMPO, varName,varBinsDictionary,
                 altitude_levels,
                 altitude_data,altitude_bins):
    
    varData_NSSL = GetVarData(data_NSSL, varName)
    varData_TEMPO = GetVarData(data_TEMPO, varName)

    #Getting Histogram Bins for Variable
    varBins = varBinsDictionary[varName]
    
    #Running
    [hist2d_raw_NSSL, field_edges,z_edges] = MakeHistogram(varData_NSSL,varBins, 
                                                           altitude_levels,
                                                           altitude_data,altitude_bins)
    [hist2d_raw_TEMPO, _,_] = MakeHistogram(varData_TEMPO,varBins, 
                                            altitude_levels,
                                            altitude_data,altitude_bins)

    return hist2d_raw_NSSL,hist2d_raw_TEMPO, field_edges,z_edges

In [ ]:
max_zf = np.where(ModelData_NSSL.zf<=15)[0][-1]
max_zc = np.where(ModelData_NSSL.zc<=15)[0][-1]

In [ ]:
def GetAltitudeThings(ModelData,varName):
    
    #Get Altitude Bins
    if varName == "w":
        altitude_levels = ModelData.zf[0:max_zf+1]
        altitude_bins   = ModelData.zc[0:max_zc+1]
    else:
        altitude_levels = ModelData.zc[0:max_zf+1]
        altitude_bins   = ModelData.zf[0:max_zc+1]
    
    ny, nx = len(ModelData.latitude),len(ModelData.longitude)
    altitude_data = np.repeat(altitude_levels[:, None, None], ny, axis=1)
    altitude_data = np.repeat(altitude_data, nx, axis=2)

    return altitude_bins,altitude_levels,altitude_data

In [ ]:
def MakePlot(
    hist2d_NSSL, hist2d_TEMPO,
    field_edges, z_edges,
    varName, multiplier=1,
    plotType="contour"
):
    """
    Plot CFAD-style 2D histograms for NSSL and TEMPO schemes side-by-side,
    with properly aligned axes and shared colorbar.
    """

    # ------------------------------------------------------
    # 1. Compute bin centers (for contourf)
    # ------------------------------------------------------
    var_centers = 0.5 * (field_edges[:-1] + field_edges[1:])
    z_centers   = 0.5 * (z_edges[:-1] + z_edges[1:])

    # ------------------------------------------------------
    # 2. Define color levels and figure layout
    # ------------------------------------------------------
    vmin, vmax = 0, 100
    levels = np.linspace(vmin, vmax, 40)

    # Use constrained_layout so colorbar doesn’t overlap
    fig = plt.figure(figsize=(15, 6), constrained_layout=True)
    gs = GridSpec(1, 3, figure=fig, width_ratios=[1, 1, 0.05])

    # ------------------------------------------------------
    # 3. Plot NSSL
    # ------------------------------------------------------
    ax1 = fig.add_subplot(gs[0])
    if plotType == "mesh":
        p1 = ax1.pcolormesh(
            multiplier * field_edges, z_edges,
            hist2d_NSSL.T, cmap="turbo", vmin=vmin, vmax=vmax, shading="auto"
        )
    elif plotType == "contour":
        p1 = ax1.contourf(
            multiplier * var_centers, z_centers,
            hist2d_NSSL.T, levels=levels, cmap="turbo", extend="both"
        )
    ax1.set_title(f"NSSL", fontsize=13)
    ax1.set_xlabel(f"{varName} values", fontsize=11)
    ax1.set_ylabel("Altitude (km)", fontsize=11)

    # ------------------------------------------------------
    # 4. Plot TEMPO
    # ------------------------------------------------------
    ax2 = fig.add_subplot(gs[1])
    if plotType == "mesh":
        p2 = ax2.pcolormesh(
            multiplier * field_edges, z_edges,
            hist2d_TEMPO.T, cmap="turbo", vmin=vmin, vmax=vmax, shading="auto"
        )
    elif plotType == "contour":
        p2 = ax2.contourf(
            multiplier * var_centers, z_centers,
            hist2d_TEMPO.T, levels=levels, cmap="turbo", extend="both"
        )
    ax2.set_title(f"TEMPO", fontsize=13)
    ax2.set_xlabel(f"{varName} values", fontsize=11)
    ax2.set_ylabel("Altitude (km)", fontsize=11)

    # ------------------------------------------------------
    # 5. Shared y-limit and consistent grid style
    # ------------------------------------------------------
    for ax in [ax1, ax2]:
        ax.set_ylim(0, 15)

    # ------------------------------------------------------
    # 6. Shared colorbar (placed in 3rd GridSpec column)
    # ------------------------------------------------------
    cax = fig.add_subplot(gs[2])
    cbar = fig.colorbar(p2, cax=cax)
    cbar.set_label("Normalized Frequency (%)", fontsize=11)

    return fig


In [ ]:
############################
#ACCUMULATING ALL TIMESTEPS FUNCTIONS

In [ ]:
def GetFileNamePath(ModelData,varName):
    
    # Build file name
    fileName = (
        f"VariableHistograms_{varName}_{ModelData.region}_"
        f"{ModelData.case}_spinup{ModelData.spinup_hours}hrs.pkl"
    )
    
    # Build directory for radar timeseries
    outputDir = os.path.join(
        DirectoryManager.GetOutputDirectory(codeType, dataType),
        "VariableHistograms"
    )

    os.makedirs(outputDir, exist_ok=True)
    
    # Full path to the .pkl file
    fileNamePath = os.path.join(outputDir, fileName)
    return fileNamePath

In [ ]:
def RunCalculations(fileNamePath,
                    ModelData_NSSL,ModelData_TEMPO,varName,
                    DirectoryManager):
    # ----------------------------------------------------------
    # 1. LOAD EXISTING FILE IF PRESENT
    # ----------------------------------------------------------
    if os.path.exists(fileNamePath):
        print(f"Loading precomputed CFADs from: {fileNamePath}")
        with open(fileNamePath, "rb") as f:
            return pickle.load(f)

    print("CFAD file not found. Computing CFADs from scratch...")

    # ----------------------------------------------------------
    # 2. SETUP
    # ----------------------------------------------------------
    hist2d_raw_NSSL  = None
    hist2d_raw_TEMPO = None
    
    # Determine which times to process
    # time_range=range(60, 61 + 1)
    time_range = None
    times = time_range if time_range is not None else range(ModelData_NSSL.Ntime)
    
    if ModelData_NSSL.region in ["TRACER","Hawaii"]:
        mask = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_NSSL)
    # elif ModelData_NSSL.region == "PRECIP":
    
    # Loading zlevels
    z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
    zlevels = np.loadtxt(z_levels_filePath)/1e3

    # Altitude Setup
    altitude_bins,altitude_levels,altitude_data = GetAltitudeThings(ModelData_NSSL,varName)
    
    # ----------------------------------------------------------
    # 3. LOOP OVER TIME
    # ----------------------------------------------------------
    for t in tqdm(times, desc="Computing"):
        # ------------------------------
        # LOAD DATA
        # ------------------------------
        data_NSSL = ModelData_NSSL.GetDataTimestep(t, printout=False).isel(nVertLevels=slice(0,max_zc+1),nVertLevelsP1=slice(0,max_zf+1))
        data_TEMPO = ModelData_TEMPO.GetDataTimestep(t, printout=False).isel(nVertLevels=slice(0,max_zc+1),nVertLevelsP1=slice(0,max_zf+1))
    
        data_NSSL  = data_NSSL.where(mask)
        data_TEMPO = data_TEMPO.where(mask)
    
        # ------------------------------
        # CALCULATING HISTOGRAM
        # ------------------------------
        [hist2d_raw_NSSL,hist2d_raw_TEMPO, field_edges,z_edges] = RunHistogram(data_NSSL,data_TEMPO, varName,varBinsDictionary,
                                                                               altitude_levels,
                                                                               altitude_data,altitude_bins)
    
        # ------------------------------
        # ACCUMULATE
        # ------------------------------
        if hist2d_raw_NSSL is None:
            hist2d_raw_NSSL  = hist2d_raw_NSSL.copy()
            hist2d_raw_TEMPO = hist2d_raw_TEMPO.copy()
        else:
            hist2d_raw_NSSL  += hist2d_raw_NSSL
            hist2d_raw_TEMPO += hist2d_raw_TEMPO
    
    # ----------------------------------------------------------
    # 4. NORMALIZE
    # ----------------------------------------------------------
    hist2d_NSSL  = NormalizeHistogram(total_Histogram_NSSL)
    hist2d_TEMPO = NormalizeHistogram(total_Histogram_TEMPO)
    
    # ----------------------------------------------------------
    # COMBINING INTO DICTIONARY
    # ----------------------------------------------------------
    results = {
        # --- RAW (unnormalized) CFADs ---
        "hist2d_raw_NSSL":  hist2d_raw_NSSL,
        "hist2d_raw_TEMPO": hist2d_raw_TEMPO,
    
        # --- NORMALIZED CFADs ---
        "hist2d_NSSL":  hist2d_NSSL,
        "hist2d_TEMPO": hist2d_TEMPO,
    
        # --- AXES ---
        "field_edges": field_edges,
        "z_edges": z_edges}
    
    # ----------------------------------------------------------
    # SAVE TO PKL
    # ----------------------------------------------------------
    # with open(fileNamePath, "wb") as f:
    #     pickle.dump(results, f)
    
    # print(f"Saved CFADs to: {fileNamePath}")
    
    return results

In [ ]:
numBins = 40 + 1
varBinsDictionary = {"w": np.linspace(-20,35,numBins),
                     # "theta": np.linspace(280,350,numBins),
                     # "relhum": np.linspace(0,100,numBins),
                     "qv": np.linspace(0,22/1e3,numBins), 
                     "qc+qi": np.linspace(0,8/1e3,numBins),
                     "qg": np.linspace(0,10/1e3,numBins),
                     "qr": np.linspace(0,10/1e3,numBins)}

In [ ]:
# #TESTING

# t=60
# data_NSSL = ModelData_NSSL.GetDataTimestep(t, printout=False)
# data_TEMPO = ModelData_TEMPO.GetDataTimestep(t, printout=False)
# varName = 'w'

# altitude_bins,altitude_levels,altitude_data = GetAltitudeThings(ModelData_NSSL,varName)


# [hist2d_raw_NSSL,hist2d_raw_TEMPO, field_edges,z_edges] = RunHistogram(data_NSSL,data_TEMPO, varName,varBinsDictionary,
#                                                                        altitude_levels,
#                                                                        altitude_data,altitude_bins)
# hist2d_NSSL = NormalizeHistogram(hist2d_raw_NSSL)
# hist2d_TEMPO = NormalizeHistogram(hist2d_raw_TEMPO)

# fig = MakePlot(
#     hist2d_NSSL, hist2d_TEMPO,
#     field_edges, z_edges,
#     varName="w",
#     multiplier=1,
#     plotType="contour"
# )

In [ ]:
varNames = ["w","qv","qc+qi","qg","qr"]
varName = varNames[0]

fileNamePath = GetFileNamePath(ModelData_NSSL,varName)
results = RunCalculations(fileNamePath,
                          ModelData_NSSL,ModelData_TEMPO,varName,
                          DirectoryManager)

In [ ]:
###########################
#PLOTTING

In [ ]:
hist2d_raw_NSSL = results["hist2d_raw_NSSL"]
hist2d_raw_TEMPO = results["hist2d_raw_TEMPO"]
hist2d_NSSL = results["hist2d_NSSL"]
hist2d_TEMPO = results["hist2d_TEMPO"]
field_edges = results["field_edges"]
z_edges = results["z_edges"]

In [ ]:
fig = MakePlot(
    hist2d_NSSL, hist2d_TEMPO,
    field_edges, z_edges,
    varName="w",
    multiplier=1,
    plotType="contour"
)